<a href="https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The chief goal is to provide decision-makers with information about which pages should be refreshed. In the ML model (Gradient Boosting), content items with higher predicted probabilities are to be reviewed first.

| Reason code | Meaning |
|---|---|
| REFRESH_PRIORITY | High probability that the content requires updating |
| TRAFFIC_DECLINE | Recent performance signals indicate declining visibility |
| STALE_CONTENT | Content has not been updated recently |
| LOW_CONFIDENCE | Model prediction requires manual validation |


# Archetype → Action Mapping

| Content archetype | Observed signal | Recommended action |
|---|---|---|
| Stale content | High days since update | Review for refresh |
| Declining content | Lower recent traffic | Analyze performance drop |
| Valuable old content | Old but strong engagement | Prioritize refresh |
| Low-value content | Low traffic and engagement | Monitor only |

As far as the SEO team is concerned, they need to pay more attention to the most relevant content opportunities first.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

url = "https://raw.githubusercontent.com/estuchis21/ml-internship-week1/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df.columns


df["refresh_flag"] = (
    (df["trend_pct"] < -0.20) &
    (df["avg_position"] > 5)
).astype(int)


df["refresh_flag"].value_counts()

y = df["refresh_flag"]

numeric_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns




features = [
    "days_with_impressions",
    "word_count",
    "char_count",
    "days_since_last_update",
    "days_with_sessions",
    "ai_sessions_90d",
    "pageviews_90d",
    "users_90d",
    "sessions_90d",
    "clicks_90d",
    "impressions_90d",
    "clicks_last_30d",
    "impressions_last_30d",
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]


numeric_features = df[features].select_dtypes(
    include=["int64","float64"]
).columns


correlation_features = (
    df[list(numeric_features) + ["refresh_flag"]]
    .corr()["refresh_flag"]
    .drop("refresh_flag")
    .sort_values(
        ascending=False
    )
)


X = df[features]

y = df["refresh_flag"]

correlation_features
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())



# ============================
# TRY DIFFERENT RANDOM FORESTS
# ============================
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.inspection import permutation_importance
rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight={
        0:1,
        1:2
    },
    random_state=42,
    n_jobs=-1
)


rf.fit(
    X_train,
    y_train
)


from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier


gb = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "model",
        GradientBoostingClassifier(
            n_estimators=500,
            learning_rate=0.005,
            max_depth=6,
            random_state=42
        )
    )

])


gb.fit(
    X_train,
    y_train
)

y_pred = gb.predict(
    X_test
)


y_prob = gb.predict_proba(
    X_test
)[:,1]


from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


y_pred = gb.predict(X_test)

y_prob = gb.predict_proba(X_test)[:,1]


results_scores = pd.DataFrame({

    "Model": [
        "Gradient Boosting"
    ],

    "Accuracy": [
        accuracy_score(
            y_test,
            y_pred
        )
    ],

    "Precision": [
        precision_score(
            y_test,
            y_pred
        )
    ],

    "Recall": [
        recall_score(
            y_test,
            y_pred
        )
    ],

    "F1": [
        f1_score(
            y_test,
            y_pred
        )
    ],

    "ROC-AUC": [
        roc_auc_score(
            y_test,
            y_prob
        )
    ]
})

print(results_scores)

results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted"] = y_pred

false_negatives = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

print(false_negatives.head(10))

false_positives = results[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1)
]

print(false_positives.head(10))

results

# Create ranking table

queue = X_test.copy()

queue["actual_refresh"] = y_test.values

queue["refresh_probability"] = y_prob


# Rank highest probability first

queue = queue.sort_values(
    "refresh_probability",
    ascending=False
)


queue.head(10)

def assign_reason(row):

    if row["days_since_last_update"] > 180:
        return "STALE_CONTENT"

    elif row["clicks_last_30d"] < row["clicks_prev_30d"]:
        return "TRAFFIC_DECLINE"

    elif row["refresh_probability"] > 0.8:
        return "REFRESH_PRIORITY"

    else:
        return "LOW_CONFIDENCE"



queue["reason_code"] = queue.apply(
    assign_reason,
    axis=1
)


queue[
    [
        "refresh_probability",
        "reason_code",
        "days_since_last_update",
        "clicks_last_30d",
        "clicks_prev_30d"
    ]
].head(20)



(24000, 25)
(6000, 25)
refresh_flag
1    13698
0    10302
Name: count, dtype: int64
refresh_flag
1    3425
0    2575
Name: count, dtype: int64
               Model  Accuracy  Precision    Recall        F1   ROC-AUC
0  Gradient Boosting  0.859333   0.819827  0.965839  0.886863  0.922603
       days_with_impressions  word_count  char_count  days_since_last_update  \
29033                     88         NaN         NaN                      22   
5842                      88      2990.0     18321.0                     104   
28858                     88         NaN         NaN                      35   
14797                     88         NaN         NaN                     104   
3559                      88         NaN         NaN                      20   
20789                     88      2817.0     16903.0                      26   
29800                     88         NaN         NaN                     104   
7124                      88      2709.0     16469.0                     

,refresh_probability,reason_code,days_since_last_update,clicks_last_30d,clicks_prev_30d
20347,0.895937,REFRESH_PRIORITY,28,0,0
21190,0.889803,REFRESH_PRIORITY,8,0,0
15324,0.888867,REFRESH_PRIORITY,104,0,0
3363,0.887326,TRAFFIC_DECLINE,104,0,1
28366,0.886448,REFRESH_PRIORITY,28,0,0
24130,0.886053,REFRESH_PRIORITY,22,0,0
23961,0.885868,REFRESH_PRIORITY,8,0,0
10019,0.884944,REFRESH_PRIORITY,104,0,0
161,0.884260,REFRESH_PRIORITY,104,0,0
10116,0.883914,REFRESH_PRIORITY,20,0,0


## 2. Intended use
This model is used for a SEO team as a decision-making tool. They will need to prioritize the content items which should be reviewed first.

This model uses signals such as traffic performance, traffic age, engagement metrics such as CTR, and the page historical visibility.

# Limitations
1. It recognises patterns related to past data, wherein it is challenging to say whether it will perform as well as now, in the future.
2. In the future, it will be hard to say whether there will be missing or inaccurate data.
3. It is vital that a SEO team review this model, for a human judgement.
4. Performance may change depending on the page or content item.

All things considered, this model should come in handy as a tool, not as something with absolute truth.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

The model should be reviewed before any content refresh decision is made.

The reviewer shoud check:
1. New content items: to verify the metrics of each new prediction.
2. Data quality: whether there are any inaccuracy or missing data.
3. Context: be it for user interests, new content or business aims.

As I said, this model aims to provide prioritization, but final decisions must be made by a human reviewer.

## no-go list
1. Automatically changing any data without human approval.
2. Automatically write or delete pages without human approval.
3. Automatically deciding content quality without reviewing the actual page manually.
4. Applying pre-processing without validation, for the sake of data quality and accuracy.

## 4. Monitoring / retrain triggers

It is vital for a SEO team to monitor the model to detect when data becomes outdated or inaccurate or less reliable.

The following situations may indicate that there needs reviewing.
1. The content distribution changes significantly, for example, a large increase in new content items or changes in traffic patterns.

2. The relationship between model predictions and actual refresh outcomes changes.

3. The model starts generating many recommendations that are not considered useful by SEO reviewers.

# Refresh recommendation monitoring

If a content item becomes stale but has no meaningful engagement signals, it may not represent a valuable refresh opportunity. In this case, the recommendation should be reviewed carefully before investing resources.

If a content item is stale but still receives engagement, traffic, or impressions, the recommendation should receive higher priority. The model prediction should be reviewed together with content quality and business context to determine whether refreshing the page is appropriate.

## Retraining triggers

The model should be considered for retraining when:

- New historical data becomes available.
- Content performance patterns change.
- Model performance decreases on recent validation data.
- Human reviewers consistently disagree with model recommendations.

Retraining should improve alignment between model predictions and current SEO
decision-making needs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

The model outputs are exported to `work/outputs/` so they can be reused for
reporting and analysis.

The exported files include:

- A ranked refresh opportunity queue containing the highest priority content
  items according to model predictions.

- Model evaluation metrics used to summarize observed performance.

- Supporting figures that help communicate model behavior.

These exports provide reproducible inputs for the final paper and avoid relying
only on notebook outputs.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os



os.makedirs(
    "../outputs",
    exist_ok=True
)

queue.to_csv(
    "../outputs/refresh_queue.csv",
    index=False
)
results_scores.to_csv(
    "../outputs/model_metrics.csv",
    index=False
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.